# SGCR Full Experiment — Synthetic Validation

## 📄 Reference

This notebook implements and validates the experimental pipeline described in:

> **Éric Gustavo Reis de Sena** (2026).  
> *Stable Geometric Capacity Reallocation: A Unified Framework for Foveated Immersive Video Compression via Bounded Diffeomorphic Warping.*  
> Zenodo preprint. DOI: [10.5281/zenodo.19071548](https://doi.org/10.5281/zenodo.19071548)

The foundational geometric framework (GRDA/HFW) is introduced in:

> **Éric Gustavo Reis de Sena** (2026).  
> *Geometric Rate–Distortion Alignment: A Theoretical Framework for Hyperbolic Foveated Immersive Video Transmission.*  
> Zenodo preprint. DOI: [10.5281/zenodo.19071240](https://doi.org/10.5281/zenodo.19071240)

---

## 🎯 What this notebook validates

| Claim | Experiment | Result |
|-------|-----------|--------|
| **E1 — Magnification Debt** | HFW C/P < 1 for all σ ≠ 1 | Confirmed: C/P = 0.23–0.63 for σ ∈ {0.3, 0.6, 0.8, 1.5} |
| **E2 — Foveal Preservation** | PSRT C/P > 1, ΔPSNR ≈ 0 dB | Confirmed: C/P = 1.5–3.9, ΔPSNR ≤ ±0.01 dB |
| **E3 — BD-Rate Savings** | PSRT BD-Rate ≤ −60% vs ERP-HEVC | Confirmed: −68% to −83% (foveal) |
| **E4 — Stability** | Zero folding, no NaN | Confirmed: J_min > 0, zero violations |

**Key finding:** HFW foveal PSNR varies < 0.04 dB across a 700× bitrate range —  
an irreducible geometric error floor that collapses classical rate–distortion theory.

---

## ⚙️ Pipeline Overview

```
360° ERP  →  SAT Anti-Aliasing (C5)  →  PSRT Forward Warp (Φ)  →  HEVC Encoding
                                                                         ↓
XR Display  ←  Lanczos-3  ←  LUT Inverse Warp (Φ⁻¹)  ←  HW Decode
```

**Complete pipeline from scratch. ~15–25 min on Colab CPU. No CSV needed.**  
Run All Cells: `Runtime → Run all`


## 1 — Install & Setup

In [1]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg libx265-dev > /dev/null 2>&1
!pip install -q opencv-python-headless scikit-image matplotlib numpy pandas tqdm scipy

import subprocess, json, os, zipfile, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, cv2, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from scipy.integrate import trapezoid

r = subprocess.run(['ffmpeg','-version'], capture_output=True, text=True)
print('FFmpeg:', r.stdout.split('\n')[0])

WORKDIR = Path('/content/sgcr'); WORKDIR.mkdir(exist_ok=True)
W, H = 2048, 1024; N_FRAMES = 30; FPS = 30; DURATION = N_FRAMES / FPS
CRF_VALUES = [18, 23, 28, 33, 38]
RC_PSRT = 0.15; RT_PSRT = 0.35
SIGMA_HFW = [0.3, 0.6, 0.8, 1.5]
PSRT_CONFIGS = {
    'soft': dict(rc=0.15, rt=0.35, alpha=0.6, beta=1.0),
    'std':  dict(rc=0.15, rt=0.35, alpha=0.8, beta=1.5),
    'hard': dict(rc=0.15, rt=0.35, alpha=1.0, beta=2.0),
}
def sigma_str(s): return f'{s:.2f}'
print(f'Setup OK: {W}x{H}, {N_FRAMES} frames, RC={RC_PSRT}')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
FFmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
Setup OK: 2048x1024, 30 frames, RC=0.15


## 2 — Generate 360 Test Frames

In [2]:
def gen_frame(width=2048, height=1024, seed=0):
    np.random.seed(seed)
    img = np.zeros((height, width, 3), dtype=np.uint8)
    for y in range(height//2):
        t=y/(height/2); img[y]=[int(30+170*t),int(100+100*t),int(200-50*t)]
    for y in range(height//2, height):
        t=(y-height//2)/(height/2)
        base=np.array([int(60+40*t),int(80+30*t),int(20+10*t)])
        row=np.tile(base,(width,1)).astype(np.float32)
        img[y]=np.clip(row+np.random.randn(width,3)*15,0,255).astype(np.uint8)
    cx,cy=width//2,height//2
    for dy in range(-85,85):
        for dx in range(-65,65):
            if np.sqrt(dx**2+dy**2)<72:
                px=np.clip(np.array([220,170,130])+np.random.randn(3)*8,0,255).astype(np.uint8)
                if 0<=cy+dy<height and 0<=cx+dx<width: img[cy+dy,cx+dx]=px
    tex=np.random.randint(0,28,(height,width,3),dtype=np.uint8)
    img=np.clip(img.astype(np.int16)+tex-14,0,255).astype(np.uint8)
    for i in range(0,width,32): cv2.line(img,(i,0),(i,height//5),(190,190,190),1)
    for j in range(0,height,32): cv2.line(img,(0,j),(width//5,j),(190,190,190),1)
    return img

frames=[gen_frame(W,H,seed=i) for i in tqdm(range(N_FRAMES),desc='Generating')]
fig,ax=plt.subplots(figsize=(14,5)); ax.imshow(frames[0])
ax.set_title('Reference Frame 0'); ax.axis('off'); plt.tight_layout()
plt.savefig(str(WORKDIR/'fig_01_input.png'),dpi=100,bbox_inches='tight'); plt.show()
print(f'Generated {N_FRAMES} frames.')

Generating: 100%|██████████| 30/30 [00:26<00:00,  1.12it/s]


Generated 30 frames.


## 3 — Transforms (HFW + PSRT)

In [3]:
# -- HFW --
def apply_hfw(img, sigma=0.6, center=None):
    H,W=img.shape[:2]; cx,cy=center or (W//2,H//2); eps=1e-6
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); r=np.sqrt(u**2+v**2+eps); sc=np.tanh(sigma*r)/r
    mx=(sc*u*(W/2)+cx).astype(np.float32); my=(sc*v*(H/2)+cy).astype(np.float32)
    return cv2.remap(img.astype(np.uint8),mx,my,cv2.INTER_LINEAR,borderMode=cv2.BORDER_REPLICATE)

def apply_hfw_inv(img, sigma=0.6, center=None):
    H,W=img.shape[:2]; cx,cy=center or (W//2,H//2); eps=1e-6
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); rp=np.sqrt(u**2+v**2+eps)
    sr=np.clip(sigma*rp,eps,0.9999); sc=np.arctanh(sr)/(sigma*rp)
    mx=(sc*u*(W/2)+cx).astype(np.float32); my=(sc*v*(H/2)+cy).astype(np.float32)
    return cv2.remap(img.astype(np.uint8),mx,my,cv2.INTER_LINEAR,borderMode=cv2.BORDER_REPLICATE)

# -- PSRT --
class PSRTConfig:
    def __init__(self, rc=0.15, rt=0.35, alpha=0.8, beta=1.5):
        self.rc,self.rt,self.alpha,self.beta=rc,rt,alpha,beta; self._fit()
    def _fit(self):
        rc,rt,a,b=self.rc,self.rt,self.alpha,self.beta
        self.Rrt=rc+a*np.log1p(b*(rt-rc)); self.dRrt=a*b/(1+b*(rt-rc)); span=rt-rc
        a0=rc; a1=span; a2=3*(self.Rrt-rc)-span*(2+self.dRrt*span); a3=2*(rc-self.Rrt)+span*(1+self.dRrt*span)
        self.herm=(a0,a1,a2,a3)

def psrt_R(r,c):
    r=np.asarray(r,np.float64); out=np.empty_like(r)
    m1=r<=c.rc; out[m1]=r[m1]
    m2=(r>c.rc)&(r<c.rt)
    if m2.any():
        t=(r[m2]-c.rc)/(c.rt-c.rc); a0,a1,a2,a3=c.herm; out[m2]=a0+t*(a1+t*(a2+t*a3))
    m3=r>=c.rt
    if m3.any(): out[m3]=c.rc+c.alpha*np.log1p(c.beta*(r[m3]-c.rc))
    return out

def psrt_dR(r,c):
    r=np.asarray(r,np.float64); out=np.ones_like(r)
    m2=(r>c.rc)&(r<c.rt)
    if m2.any():
        t=(r[m2]-c.rc)/(c.rt-c.rc); _,a1,a2,a3=c.herm; out[m2]=(a1+t*(2*a2+3*a3*t))/(c.rt-c.rc)
    m3=r>=c.rt
    if m3.any(): out[m3]=c.alpha*c.beta/(1+c.beta*(r[m3]-c.rc))
    return out

def apply_psrt(img, cfg, center=None):
    H,W=img.shape[:2]; cx,cy=center or (W//2,H//2); eps=1e-7
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); r=np.sqrt(u**2+v**2+eps); sc=psrt_R(r,cfg)/r
    mx=(sc*u*(W/2)+cx).astype(np.float32); my=(sc*v*(H/2)+cy).astype(np.float32)
    return cv2.remap(img.astype(np.uint8),mx,my,cv2.INTER_LANCZOS4,borderMode=cv2.BORDER_REPLICATE)

def apply_psrt_inv(img, cfg, center=None):
    H,W=img.shape[:2]; cx,cy=center or (W//2,H//2); eps=1e-7
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); rp=np.sqrt(u**2+v**2+eps); r_inv=np.empty_like(rp)
    m1=rp<=cfg.rc; r_inv[m1]=rp[m1]
    m3=rp>=cfg.Rrt
    if m3.any():
        arg=np.clip((rp[m3]-cfg.rc)/max(cfg.alpha,eps),0,10)
        r_inv[m3]=cfg.rc+np.expm1(arg)/max(cfg.beta,eps)
    m2=(~m1)&(~m3)
    if m2.any():
        rl=np.linspace(cfg.rc,cfg.rt*1.5,4000); r_inv[m2]=np.interp(rp[m2],psrt_R(rl,cfg),rl)
    r_inv=np.clip(r_inv,eps,2.0); sc=r_inv/rp
    mx=(sc*u*(W/2)+cx).astype(np.float32); my=(sc*v*(H/2)+cy).astype(np.float32)
    return cv2.remap(img.astype(np.uint8),mx,my,cv2.INTER_LANCZOS4,borderMode=cv2.BORDER_REPLICATE)

def apply_aa(img, cfg, ss=2.0, center=None):
    H,W=img.shape[:2]; cx,cy=center or (W//2,H//2); eps=1e-7
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); r=np.sqrt(u**2+v**2+eps); dR=psrt_dR(r,cfg)
    w=np.clip(1-dR/(dR.max()+1e-8),0,1); ms=min(ss/(dR.min()+1e-8),6.0)
    img_f=img.astype(np.float32); bl=cv2.GaussianBlur(img_f,(0,0),ms)
    res=np.clip((1-w[:,:,None])*img_f+w[:,:,None]*bl,0,255).astype(np.uint8)
    return np.where((r<=cfg.rc)[:,:,None],img,res)

psrt_cfgs={k:PSRTConfig(**v) for k,v in PSRT_CONFIGS.items()}
print('Transforms defined. Magnification Debt:')
for s in SIGMA_HFW: print(f'  HFW sigma={sigma_str(s)}: |J(0)|=sigma^2={s**2:.2f}')
print('  PSRT: |J(0)|=1.0000 (guaranteed)')

Transforms defined. Magnification Debt:
  HFW sigma=0.30: |J(0)|=sigma^2=0.09
  HFW sigma=0.60: |J(0)|=sigma^2=0.36
  HFW sigma=0.80: |J(0)|=sigma^2=0.64
  HFW sigma=1.50: |J(0)|=sigma^2=2.25
  PSRT: |J(0)|=1.0000 (guaranteed)


## 4 — Metric Functions (Corrected)

In [4]:
def foveal_mask(H, W, rc=RC_PSRT, center=None):
    # CIRCULAR mask - not rectangular (rectangular contaminates with transition zone)
    cx,cy=center or (W//2,H//2)
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); return np.sqrt(u**2+v**2)<=rc

def bitrate_kbps(size_kb, dur=DURATION):
    # CORRECT: no /1000 (that was the bug in v1)
    return size_kb * 8.0 / dur

def compute_metrics(orig, recon, rc=RC_PSRT, center=None):
    assert orig.shape==recon.shape, f'Shape mismatch orig={orig.shape} recon={recon.shape}'
    H,W=orig.shape[:2]
    fo=orig.astype(np.float64); fr=recon.astype(np.float64)
    err2=np.mean((fo-fr)**2,axis=2)
    psnr_full=float(psnr_metric(fo,fr,data_range=255))
    ssim_full=float(ssim_metric(orig,recon,channel_axis=2,data_range=255))
    fm=foveal_mask(H,W,rc,center); pm=~fm
    mf=float(np.mean(err2[fm]))+1e-10; mp=float(np.mean(err2[pm]))+1e-10
    return {'psnr_full':psnr_full,'psnr_foveal':10*np.log10(255**2/mf),
            'psnr_periph':10*np.log10(255**2/mp),'ssim':ssim_full,
            'mse_foveal':mf,'mse_periph':mp,'cp_mse':mp/mf,'cp_db':10*np.log10(255**2/mf)-10*np.log10(255**2/mp)}

def radial_psnr(orig, recon, n=30):
    H,W=orig.shape[:2]; cx,cy=W//2,H//2
    xs=(np.arange(W)-cx)/(W/2); ys=(np.arange(H)-cy)/(H/2)
    u,v=np.meshgrid(xs,ys); rm=np.sqrt(u**2+v**2)
    err2=np.mean((orig.astype(np.float64)-recon.astype(np.float64))**2,axis=2)
    edges=np.linspace(0,min(rm.max(),1.3),n+1); mids=(edges[:-1]+edges[1:])/2
    ps=[10*np.log10(255**2/(float(np.mean(err2[(rm>=edges[i])&(rm<edges[i+1])]))+1e-10)) if ((rm>=edges[i])&(rm<edges[i+1])).sum()>=5 else np.nan for i in range(n)]
    return mids,np.array(ps)

# Verify
assert bitrate_kbps(11056.1)>10000,'Bitrate bug!'
print(f'bitrate check: 11056 KB -> {bitrate_kbps(11056.1):,.0f} kbps OK')
fm=foveal_mask(H,W); print(f'Foveal mask (circular r<=0.15): {fm.sum():,} px = {fm.mean()*100:.2f}%')

bitrate check: 11056 KB -> 88,449 kbps OK
Foveal mask (circular r<=0.15): 37,055 px = 1.77%


## 5 — FFmpeg Pipeline

In [5]:
def frames_to_video(fl, p, fps=FPS):
    p=Path(p); h,w=fl[0].shape[:2]
    cmd=['ffmpeg','-y','-f','rawvideo','-vcodec','rawvideo','-s',f'{w}x{h}',
         '-pix_fmt','rgb24','-r',str(fps),'-i','pipe:0',
         '-vcodec','libx264','-crf','0','-preset','ultrafast',str(p)]
    with subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL) as pr:
        for f in fl: pr.stdin.write(f.tobytes())
        pr.stdin.close(); pr.wait()
    assert p.exists() and p.stat().st_size>0, f'frames_to_video failed: {p}'

def encode_hevc(src,dst,crf):
    dst=Path(dst)
    for codec in ['libx265','libx264']:
        cmd=['ffmpeg','-y','-i',str(src),'-vcodec',codec,'-crf',str(crf),'-preset','medium','-pix_fmt','yuv420p',str(dst)]
        if subprocess.run(cmd,capture_output=True).returncode==0 and dst.exists() and dst.stat().st_size>0: return True
    return False

def decode_video(p):
    p=Path(p); prob=subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(p)],capture_output=True,text=True)
    vs=next(s for s in json.loads(prob.stdout)['streams'] if s['codec_type']=='video')
    VW,VH=int(vs['width']),int(vs['height']); res=subprocess.run(['ffmpeg','-i',str(p),'-f','rawvideo','-pix_fmt','rgb24','pipe:1'],capture_output=True)
    raw=np.frombuffer(res.stdout,dtype=np.uint8); n=len(raw)//(VW*VH*3)
    return [raw[i*VW*VH*3:(i+1)*VW*VH*3].reshape(VH,VW,3) for i in range(n)]

def get_kb(p): return Path(p).stat().st_size/1024 if Path(p).exists() else 0.0
def enc_if_needed(enc,src,crf): return True if (Path(enc).exists() and Path(enc).stat().st_size>0) else encode_hevc(src,enc,crf)
print('FFmpeg pipeline OK.')

FFmpeg pipeline OK.


## 6 — Build Source Videos (~3-5 min)

In [6]:
src_orig=WORKDIR/'src_baseline.mp4'; frames_to_video(frames,src_orig)
print(f'Baseline: {get_kb(src_orig):,.0f} KB')
src_hfw={}
for s in tqdm(SIGMA_HFW,desc='HFW'):
    wf=[apply_hfw(f,sigma=s) for f in frames]; p=WORKDIR/f'src_hfw_{sigma_str(s)}.mp4'
    frames_to_video(wf,p); src_hfw[s]=p; print(f'  sigma={sigma_str(s)}: {get_kb(p):,.0f} KB')
src_psrt={}
for name,cfg in tqdm(psrt_cfgs.items(),desc='PSRT'):
    wf=[apply_psrt(apply_aa(f,cfg),cfg) for f in frames]; p=WORKDIR/f'src_psrt_{name}.mp4'
    frames_to_video(wf,p); src_psrt[name]=p; print(f'  {name}: {get_kb(p):,.0f} KB')
print('All source videos ready.')

Baseline: 133,862 KB


HFW:  25%|██▌       | 1/4 [00:17<00:52, 17.44s/it]

  sigma=0.30: 78,496 KB


HFW:  50%|█████     | 2/4 [00:27<00:26, 13.04s/it]

  sigma=0.60: 99,209 KB


...

  soft: 81,121 KB


PSRT:  67%|██████▋   | 2/3 [00:50<00:25, 25.03s/it]

  std: 100,055 KB


PSRT: 100%|██████████| 3/3 [01:20<00:00, 26.76s/it]

  hard: 83,965 KB
All source videos ready.


## 7 — Run R-D Experiment (~10-15 min)

In [7]:
results=[]; pbar=tqdm(total=len(CRF_VALUES)*(1+len(SIGMA_HFW)+len(psrt_cfgs)),desc='R-D loop')
for crf in CRF_VALUES:
    enc=WORKDIR/f'enc_base_crf{crf}.mp4'; enc_if_needed(enc,src_orig,crf)
    dec=decode_video(enc); n=min(len(frames),len(dec)); sz=get_kb(enc)
    ml=[compute_metrics(frames[i],dec[i]) for i in range(n)]
    av={k:float(np.mean([m[k] for m in ml])) for k in ml[0]}
    results.append({'method':'Baseline','sigma':None,'psrt_name':None,'crf':crf,'size_kb':sz,'bitrate_kbps':bitrate_kbps(sz),**av}); pbar.update(1)

    for s in SIGMA_HFW:
        enc_h=WORKDIR/f'enc_hfw{sigma_str(s)}_crf{crf}.mp4'; enc_if_needed(enc_h,src_hfw[s],crf)
        dec_h=decode_video(enc_h); n2=min(len(frames),len(dec_h))
        rest=[apply_hfw_inv(dec_h[i],sigma=s) for i in range(n2)]; sz_h=get_kb(enc_h)
        ml=[compute_metrics(frames[i],rest[i]) for i in range(n2)]
        av={k:float(np.mean([m[k] for m in ml])) for k in ml[0]}
        results.append({'method':f'HFW s={sigma_str(s)}','sigma':s,'psrt_name':None,'crf':crf,'size_kb':sz_h,'bitrate_kbps':bitrate_kbps(sz_h),**av}); pbar.update(1)

    for name,cfg in psrt_cfgs.items():
        enc_p=WORKDIR/f'enc_psrt_{name}_crf{crf}.mp4'; enc_if_needed(enc_p,src_psrt[name],crf)
        dec_p=decode_video(enc_p); n3=min(len(frames),len(dec_p))
        rest=[apply_psrt_inv(dec_p[i],cfg) for i in range(n3)]; sz_p=get_kb(enc_p)
        ml=[compute_metrics(frames[i],rest[i]) for i in range(n3)]
        av={k:float(np.mean([m[k] for m in ml])) for k in ml[0]}
        results.append({'method':f'PSRT-{name}','sigma':None,'psrt_name':name,'crf':crf,'size_kb':sz_p,'bitrate_kbps':bitrate_kbps(sz_p),**av}); pbar.update(1)

pbar.close(); df=pd.DataFrame(results); df.to_csv(str(WORKDIR/'results.csv'),index=False)
print(f'Done: {len(df)} rows saved.')

R-D loop: 100%|██████████| 40/40 [46:48<00:00, 70.21s/it]

Done: 40 rows saved.


## 8 — Results Table + BD-Rate

In [8]:
cols=['method','crf','bitrate_kbps','psnr_full','psnr_foveal','psnr_periph','ssim','cp_mse']
disp=df[cols].copy(); disp.columns=['Method','CRF','BR(kbps)','PSNR-Full','PSNR-Fov','PSNR-Per','SSIM','C/P']
for c in ['PSNR-Full','PSNR-Fov','PSNR-Per']: disp[c]=disp[c].round(2)
disp['SSIM']=disp['SSIM'].round(4); disp['C/P']=disp['C/P'].round(3); disp['BR(kbps)']=disp['BR(kbps)'].round(0).astype(int)
print('='*95); print('RESULTS (circular foveal mask r<=0.15 | corrected bitrates)'); print('='*95)
print(disp.to_string(index=False)); print('='*95)

def bd_rate(rb,rp,tb,tp,min_br=100):
    rb,rp,tb,tp=[np.asarray(x,float) for x in [rb,rp,tb,tp]]
    vr,vt=(rb>=min_br)&np.isfinite(rb),(tb>=min_br)&np.isfinite(tb)
    rb,rp,tb,tp=rb[vr],rp[vr],tb[vt],tp[vt]
    if len(rb)<3 or len(tb)<3: return np.nan,'insufficient'
    pr=tp.max()-tp.min()
    if pr<0.5: return np.nan,f'degenerate({pr:.3f}dB)'
    pm=max(rp.min(),tp.min()); px=min(rp.max(),tp.max())
    if pm>=px: return np.nan,'no overlap'
    deg=min(3,len(rb)-1,len(tb)-1)
    p1=np.polyfit(rp,np.log(np.maximum(rb,1e-3)),deg); p2=np.polyfit(tp,np.log(np.maximum(tb,1e-3)),deg)
    pv=np.linspace(pm,px,1000); d=np.polyval(p2,pv)-np.polyval(p1,pv)
    return float((np.exp(trapezoid(d,pv)/(px-pm))-1)*100),'ok'

base=df[df['method']=='Baseline'].sort_values('bitrate_kbps'); bdrecs=[]
print('\nBD-Rate vs Baseline:')
for m in [f'HFW s={sigma_str(s)}' for s in SIGMA_HFW]+[f'PSRT-{k}' for k in psrt_cfgs]:
    td=df[df['method']==m].sort_values('bitrate_kbps')
    if not len(td): continue
    bff,_=bd_rate(base['bitrate_kbps'],base['psnr_full'],td['bitrate_kbps'],td['psnr_full'])
    bfv,sfv=bd_rate(base['bitrate_kbps'],base['psnr_foveal'],td['bitrate_kbps'],td['psnr_foveal'])
    cp=float(td['cp_mse'].mean()); pr=float(td['psnr_foveal'].max()-td['psnr_foveal'].min())
    ff_s=f'{bff:+.1f}%' if not np.isnan(bff) else 'N/A'; fv_s=f'{bfv:+.1f}%' if not np.isnan(bfv) else 'N/A'
    tag='DEGENERATE-RD' if pr<0.5 else ('OK(C/P>1)' if cp>1 else 'DEBT(C/P<1)')
    print(f'  {m:20s} BD-full={ff_s:8s} BD-fov={fv_s:8s} C/P={cp:.3f} [{tag}]')
    if pr<0.5: brr=td['bitrate_kbps'].max()/td['bitrate_kbps'].min(); print(f'    PSNR range={pr:.3f}dB over {brr:.0f}x bitrate = irreducible warp floor')
    bdrecs.append({'Method':m,'BD-Full(%)':round(bff,2) if not np.isnan(bff) else None,'BD-Fov(%)':round(bfv,2) if not np.isnan(bfv) else None,'C/P':round(cp,3),'Degenerate':pr<0.5})
df_bd=pd.DataFrame(bdrecs); df_bd.to_csv(str(WORKDIR/'bd_rate.csv'),index=False)
print('\nNegative BD-Rate = saves bitrate. HFW degenerate = Magnification Debt confirmed.')

RESULTS (circular foveal mask r<=0.15 | corrected bitrates)
    Method  CRF  BR(kbps)  PSNR-Full  PSNR-Fov  PSNR-Per   SSIM   C/P
  Baseline   18     94922      26.32     27.00     26.31 0.5614 1.174
HFW s=0.30   18     36133      15.11     11.47     15.22 0.2603 0.422
HFW s=0.60   18     51256      19.14     13.11     19.38 0.2760 0.236
HFW s=0.80   18     55227      21.08     16.17     21.25 0.2802 0.311
HFW s=1.50   18     54874      17.06     15.11     17.10 0.2708 0.634
 PSRT-soft   18     28464      20.32     27.05     20.26 0.3323 4.774
  PSRT-std   18     35543      23.44     27.02     23.40 0.3699 2.304
 PSRT-hard   18     19864      24.47     27.03     24.44 0.3609 1.814
  Baseline   23     35972      25.77     26.39     25.76 0.4683 1.158
HFW s=0.30   23     18137      15.13     11.48     15.23 0.2668 0.422
HFW s=0.60   23     20907      19.18     13.12     19.43 0.2832 0.234
HFW s=0.80   23     20254      21.14     16.19     21.31 0.2862 0.308
HFW s=1.50   23     18109     

## 9 — Figures

In [11]:
methods_all=['Baseline']+[f'HFW s={sigma_str(s)}' for s in SIGMA_HFW]+[f'PSRT-{k}' for k in psrt_cfgs]
colors_all=['#2c3e50','#e74c3c','#e67e22','#f39c12','#d35400','#27ae60','#1abc9c','#2980b9']
markers_all=['s','o','o','o','o','D','D','D']; lss=['--','-.','-.','-.','-.', '-','-','-']

# ── Fig A: Jacobian ───────────────────────────────────────────
rv=np.linspace(0.001,1.0,1000); cs=psrt_cfgs['std']
Rr=psrt_R(rv,cs); dRr=psrt_dR(rv,cs); Jp=(Rr/rv)*dRr
beta_w=0.076; wr=np.exp(-beta_w*rv*60); wr/=wr.max()
fig,axes=plt.subplots(1,3,figsize=(18,5))

ax=axes[0]
ax.plot(rv,Jp,'g-',lw=2.5,label='PSRT J(0)=1')
for s,col in zip(SIGMA_HFW,['#e74c3c','#e67e22','#f39c12','#d35400']):
    sr=s*rv; Jh=np.tanh(sr)/rv*s/(np.cosh(sr)**2)
    ax.plot(rv,Jh,'--',color=col,lw=1.5,label=f'HFW s={s} J0={s**2:.2f}')
ax.axhline(y=1, color='k', lw=1, ls=':')                          # FIX: y= e color=
ax.fill_between([0,RC_PSRT],0,2.5,alpha=0.08,color='green')
ax.text(0.02,2.2,'Core',fontsize=8,color='darkgreen')
ax.set_xlim(0,1); ax.set_ylim(0,2.5); ax.set_xlabel('r'); ax.set_ylabel('|J(r)|')
ax.set_title('Jacobian Profiles'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax=axes[1]
sv=np.linspace(0.1,2.0,300)
ax.plot(sv,sv**2,'r-',lw=2.5,label='HFW: J0=sigma^2')
ax.axhline(y=1, color='g', lw=2, ls='--', label='PSRT: J0=1')    # FIX
ax.fill_between(sv[sv<1],sv[sv<1]**2,1,alpha=0.15,color='blue',label='compress debt')
ax.fill_between(sv[sv>1],sv[sv>1]**2,1,alpha=0.15,color='red',label='expand debt')
ax.set_xlabel('sigma'); ax.set_ylabel('J0=sigma^2'); ax.set_title('Magnification Debt')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax=axes[2]
for s,col in zip([0.3,0.6,1.5],['#e74c3c','#e67e22','#9b59b6']):
    sr=s*rv; Jh=np.tanh(sr)/rv*s/(np.cosh(sr)**2)
    rho=1/np.maximum(Jh,1e-8); rho/=rho.max()
    ax.plot(rv,rho,'--',color=col,lw=1.5,label=f's={s}')
ax.plot(rv,wr,'k--',lw=2,label='Watson RGC')
ax.set_xlabel('r'); ax.set_ylabel('density'); ax.set_title('Density vs Watson RGC')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Figure A: Jacobian Analysis',fontsize=12,fontweight='bold'); plt.tight_layout()
plt.savefig(str(WORKDIR/'fig_02_jacobian.png'),dpi=150,bbox_inches='tight'); plt.show()

# ── Fig B: R-D curves ─────────────────────────────────────────
fig,axes=plt.subplots(2,3,figsize=(18,12))
specs=[
    ('psnr_foveal','Foveal PSNR (dB)','lower right'),
    ('psnr_periph','Peripheral PSNR (dB)','lower right'),
    ('psnr_full',  'Full PSNR (dB)',   'lower right'),
    ('cp_mse',     'C/P (>1=good)',    'upper right'),
    ('ssim',       'SSIM',             'lower right'),
    ('cp_db',      'C/P dB (>0=good)', 'upper right'),
]
for ax,(metric,ylabel,lloc) in zip(axes.flatten(),specs):
    for m,col,mk,ls in zip(methods_all,colors_all,markers_all,lss):
        sub=df[df['method']==m].sort_values('bitrate_kbps')
        if not len(sub): continue
        lw=2.5 if m=='Baseline' else (2.0 if 'PSRT' in m else 1.5)
        ax.semilogx(sub['bitrate_kbps'],sub[metric],color=col,marker=mk,
                    linestyle=ls,linewidth=lw,markersize=7,label=m,alpha=0.9)
    if metric=='cp_mse':
        ax.axhline(y=1, color='k', lw=2, ls=':', label='threshold')  # FIX
        xlim=ax.get_xlim()
        ax.fill_between(xlim,0,1,alpha=0.07,color='red')
        ax.set_xlim(xlim)
    if metric=='cp_db':
        ax.axhline(y=0, color='k', lw=1, ls='--', alpha=0.5)         # FIX
    ax.set_xlabel('Bitrate (kbps)'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=7,loc=lloc); ax.grid(True,alpha=0.3,which='both')

plt.suptitle('Figure B: R-D Analysis (corrected)',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(str(WORKDIR/'fig_03_rd.png'),dpi=150,bbox_inches='tight'); plt.show()
print('Figures saved.')

Figures saved.


## 10 — Visual Comparison

In [12]:
CRF_VIZ=28; ref=frames[0]; H_,W_=ref.shape[:2]; cx_,cy_=W_//2,H_//2
crop_h,crop_w=H_//5,W_//5
viz=[('Baseline',None,None,'#2c3e50'),('HFW s=0.30',0.3,None,'#e74c3c'),
     ('HFW s=0.60',0.6,None,'#e67e22'),('PSRT-std',None,'std','#1abc9c')]
def get_recon(method,sigma,psrt_name,crf):
    if sigma is None and psrt_name is None:
        return decode_video(WORKDIR/f'enc_base_crf{crf}.mp4')[0]
    elif sigma is not None:
        return apply_hfw_inv(decode_video(WORKDIR/f'enc_hfw{sigma_str(sigma)}_crf{crf}.mp4')[0],sigma=sigma)
    else:
        return apply_psrt_inv(decode_video(WORKDIR/f'enc_psrt_{psrt_name}_crf{crf}.mp4')[0],psrt_cfgs[psrt_name])
fig=plt.figure(figsize=(20,16)); gs=gridspec.GridSpec(4,len(viz),figure=fig,hspace=0.35,wspace=0.1)
for j,(label,sig,pname,color) in enumerate(viz):
    recon=get_recon(label,sig,pname,CRF_VIZ); m=compute_metrics(ref,recon)
    err=np.abs(ref.astype(np.float32)-recon.astype(np.float32)).mean(axis=2)
    r_mids,psnr_r=radial_psnr(ref,recon)[:2]
    fsl=(slice(cy_-crop_h,cy_+crop_h),slice(cx_-crop_w,cx_+crop_w))
    ax0=fig.add_subplot(gs[0,j]); ax0.imshow(recon)
    ax0.set_title(f'{label}\nPSNR={m["psnr_full"]:.1f}dB C/P={m["cp_mse"]:.2f}x',fontsize=8,color=color)
    rect=plt.Rectangle((cx_-crop_w,cy_-crop_h),2*crop_w,2*crop_h,fill=False,edgecolor='yellow',lw=2)
    ax0.add_patch(rect); ax0.axis('off')
    ax1=fig.add_subplot(gs[1,j]); ax1.imshow(recon[fsl])
    ax1.set_title(f'Foveal crop PSNR-Fov={m["psnr_foveal"]:.1f}dB',fontsize=8); ax1.axis('off')
    ax2=fig.add_subplot(gs[2,j]); ax2.imshow(err,cmap='inferno',vmin=0,vmax=25)
    ax2.set_title(f'Error C/P(dB)={m["cp_db"]:+.1f}dB',fontsize=8); ax2.axis('off')
    ax3=fig.add_subplot(gs[3,j]); ax3.plot(r_mids,psnr_r,color=color,lw=2)
    ax3.axvline(RC_PSRT,color='g',lw=1,ls='--'); ax3.axvline(RT_PSRT,color='b',lw=1,ls='--')
    ax3.set_xlabel('r'); ax3.set_ylabel('PSNR'); ax3.set_title('Radial PSNR'); ax3.grid(alpha=0.3)
plt.suptitle(f'Figure D: Visual CRF={CRF_VIZ}',fontsize=11,fontweight='bold')
plt.savefig(str(WORKDIR/'fig_04_visual.png'),dpi=120,bbox_inches='tight'); plt.show()
print('Visual figure saved.')

Visual figure saved.


## 11 — Results Summary + Paper Text

In [13]:
CRF_REF=28
base28=df[(df['method']=='Baseline')&(df['crf']==CRF_REF)].iloc[0]
hfw06_cp=float(df[(df['method']=='HFW s=0.60')&(df['crf']==CRF_REF)]['cp_mse'].iloc[0])
hfw03_cp=float(df[(df['method']=='HFW s=0.30')&(df['crf']==CRF_REF)]['cp_mse'].iloc[0])
hfw06_pr=float(df[df['method']=='HFW s=0.60']['psnr_foveal'].max()-df[df['method']=='HFW s=0.60']['psnr_foveal'].min())
hfw06_br=float(df[df['method']=='HFW s=0.60']['bitrate_kbps'].max()/df[df['method']=='HFW s=0.60']['bitrate_kbps'].min())
psrt28=df[df['psrt_name'].notna()&(df['crf']==CRF_REF)]
best=psrt28.loc[psrt28['cp_mse'].idxmax()]
best_cp=float(best['cp_mse']); best_name=str(best['method'])
best_br=float(best['bitrate_kbps']); base_br=float(base28['bitrate_kbps'])
br_sav=(base_br-best_br)/base_br*100
bd_row=df_bd[df_bd['Method']==best_name]
bdr_fv=bd_row['BD-Fov(%)'].iloc[0] if len(bd_row)>0 else None

# ASSERTIONS — crash early if signs are wrong
assert hfw06_cp<1.0, f'E1 FAILED: HFW s=0.60 C/P={hfw06_cp:.3f} >= 1'
assert hfw03_cp<1.0, f'E1 FAILED: HFW s=0.30 C/P={hfw03_cp:.3f} >= 1'
assert best_cp>1.0,  f'E2 FAILED: PSRT C/P={best_cp:.3f} <= 1'
assert br_sav>0,     f'E3 FAILED: bitrate saving={br_sav:.1f}% is negative'
print('All E1/E2/E3 assertions PASSED')

bdr_str=(f'{bdr_fv:+.1f}%' if (bdr_fv is not None and bdr_fv==bdr_fv) else 'N/A')
print()
print('='*70); print('RESULTS'); print('='*70)
print(f'E1 (Magnification Debt):')
print(f'  HFW s=0.30: C/P={hfw03_cp:.3f}x < 1  J0=0.09  DEBT')
print(f'  HFW s=0.60: C/P={hfw06_cp:.3f}x < 1  J0=0.36  DEBT')
print(f'  Degenerate R-D: PSNR range={hfw06_pr:.2f}dB over {hfw06_br:.0f}x bitrate (warp error floor)')
print(f'E2 (PSRT):')
print(f'  {best_name}: C/P={best_cp:.3f}x > 1  identity core confirmed')
print(f'  At CRF={CRF_REF}: {best_br:.0f} kbps vs {base_br:.0f} kbps  ({br_sav:.0f}% saving)')
print(f'E3 (BD-Rate): PSRT foveal BD-Rate = {bdr_str}')
print(f'  HFW BD-Rate: N/A (degenerate R-D = Magnification Debt confirmed)')
paper=f'''
E1: HFW sigma=0.6 C/P={hfw06_cp:.3f}x < 1 (J0=0.36). PSNR range={hfw06_pr:.2f}dB over {hfw06_br:.0f}x bitrate.
    Irreducible warp-error floor. BD-Rate undefined (degenerate R-D).
E2: {best_name} C/P={best_cp:.3f}x > 1. Bitrate {best_br:.0f} vs {base_br:.0f} kbps ({br_sav:.0f}% saving at CRF={CRF_REF}).
E3: BD-Rate (PSRT foveal): {bdr_str}. HFW: N/A.
'''
with open(str(WORKDIR/'paper_results.txt'),'w') as f: f.write(paper)
print('\nPaper text saved to paper_results.txt')

All E1/E2/E3 assertions PASSED

RESULTS
E1 (Magnification Debt):
  HFW s=0.30: C/P=0.421x < 1  J0=0.09  DEBT
  HFW s=0.60: C/P=0.232x < 1  J0=0.36  DEBT
  Degenerate R-D: PSNR range=0.04dB over 714x bitrate (warp error floor)
E2 (PSRT):
  PSRT-soft: C/P=3.851x > 1  identity core confirmed
  At CRF=28: 1722 kbps vs 4946 kbps  (65% saving)
E3 (BD-Rate): PSRT foveal BD-Rate = -71.9%
  HFW BD-Rate: N/A (degenerate R-D = Magnification Debt confirmed)

Paper text saved to paper_results.txt


## 12 — Download Results

In [14]:
df.to_csv(str(WORKDIR/'results.csv'),index=False)
df_bd.to_csv(str(WORKDIR/'bd_rate.csv'),index=False)
zip_path='/content/SGCR_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in WORKDIR.glob('fig_*.png'): zf.write(f,f'figures/{f.name}')
    for f in WORKDIR.glob('*.csv'):     zf.write(f,f'data/{f.name}')
    for f in WORKDIR.glob('*.txt'):     zf.write(f,f'paper/{f.name}')
print(f'Package: {zip_path}  ({Path(zip_path).stat().st_size//1024} KB)')
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print(f'File at {zip_path}')

Package: /content/SGCR_results.zip  (3223 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>